comparing terrset and python appraoches for TS

In [ ]:
import numpy as np, rasterio
from pathlib import Path

main_workspace = "./data"; metric_type = "edge"     # set to what you ran
tau_path = Path(main_workspace)/"4_TS_and_TI"/"TS"/metric_type/f"{metric_type}_raster_group_MK_tau.tif"

total = n_nodata = n_zero = n_nonzero = 0
with rasterio.open(tau_path) as s:
    nodata = s.nodata
    print("nodata value:", nodata)
    for _, win in s.block_windows(1):
        a = s.read(1, window=win, masked=True)
        data = np.ma.getdata(a)
        mask = np.ma.getmaskarray(a)          # True = nodata/background
        total    += data.size
        n_nodata += int(mask.sum())
        valid = data[~mask]
        n_zero    += int((valid == 0).sum())
        n_nonzero += int((valid != 0).sum())

print(f"total pixels   : {total:,}")
print(f"NoData/NaN     : {n_nodata:,}  ({n_nodata/total:.1%})  <- background dropped")
print(f"exactly 0.0    : {n_zero:,}  ({n_zero/total:.1%})  <- REAL zero-trend pixels")
print(f"nonzero Tau    : {n_nonzero:,}  ({n_nonzero/total:.1%})  <- real trends")


In [ ]:
# compare python and terrset outputs for tau
import numpy as np, rasterio
from pathlib import Path

py_path   = Path(r"./data/4_TS_and_TI/TS_zscore/pn/pn_MK_tau_z.tif")
#main_workspace = "./data"  # Path(r"D:\GitHub\typology\data")
terr_path = Path(r"./data/4_TS_and_TI/terrset_version/pn_MK_tau_z.tif")

print("python exists :", py_path.exists())
print("terrset exists:", terr_path.exists())

with rasterio.open(py_path) as a, rasterio.open(terr_path) as b:
    print("python  grid:", a.width, "x", a.height, "| nodata:", a.nodata)
    print("terrset grid:", b.width, "x", b.height, "| nodata:", b.nodata)
    if (a.width, a.height) != (b.width, b.height):
        raise SystemExit("GRIDS DIFFER - align before comparing.")

    both = py_only = terr_only = 0
    sum_d = sum_d2 = 0.0; max_abs = 0.0
    sx=sy=sxx=syy=sxy=0.0
    n_py=0; spx=spxx=0.0; n_tr=0; stx=stxx=0.0
    for _, win in a.block_windows(1):          # memory-safe, one tile at a time
        pa = a.read(1, window=win, masked=True); pb = b.read(1, window=win, masked=True)
        va, ma = np.ma.getdata(pa).astype(np.float64), np.ma.getmaskarray(pa)
        vb, mb = np.ma.getdata(pb).astype(np.float64), np.ma.getmaskarray(pb)
        py_only   += int(((~ma) & mb).sum())
        terr_only += int((ma & (~mb)).sum())
        xp = va[~ma]; n_py += xp.size; spx += xp.sum(); spxx += (xp*xp).sum()
        xt = vb[~mb]; n_tr += xt.size; stx += xt.sum(); stxx += (xt*xt).sum()
        v = (~ma) & (~mb); x = va[v]; y = vb[v]
        if x.size:
            d = x - y; both += x.size
            sum_d += d.sum(); sum_d2 += (d*d).sum()
            mm = np.abs(d).max();  max_abs = mm if mm > max_abs else max_abs
            sx+=x.sum(); sy+=y.sum(); sxx+=(x*x).sum(); syy+=(y*y).sum(); sxy+=(x*y).sum()

print("--- footprints ---")
print(f"valid Python : {n_py:,}")
print(f"valid TerrSet: {n_tr:,}")
print(f"valid in BOTH: {both:,}")
print(f"Python-only  : {py_only:,}")
print(f"TerrSet-only : {terr_only:,}")
print("--- per-raster z-score stats (each ~0 mean, ~1 std) ---")
print(f"Python : mean={spx/n_py:+.4f} std={(spxx/n_py-(spx/n_py)**2)**0.5:.4f}")
print(f"TerrSet: mean={stx/n_tr:+.4f} std={(stxx/n_tr-(stx/n_tr)**2)**0.5:.4f}")
if both:
    r = (both*sxy - sx*sy)/(((both*sxx-sx*sx)*(both*syy-sy*sy))**0.5)
    print("--- agreement on BOTH-valid pixels ---")
    print(f"mean diff (Py-Terr): {sum_d/both:+.5f}")
    print(f"RMSE               : {(sum_d2/both)**0.5:.5f}")
    print(f"max abs diff       : {max_abs:.5f}")
    print(f"correlation r      : {r:.5f}")


TS vs TI 
using kiro

In [ ]:
#comparing agreement by TS z-score threshold
import numpy as np, rasterio
from pathlib import Path

ts_z_path = Path("./data/4_TS_and_TI/TS_zscore/area/area_MK_tau_z.tif")
ti_diff   = Path("./data/4_TS_and_TI/TI/img_diff/90-20_area.tif")

print("ts exists:", ts_z_path.exists(), "| ti exists:", ti_diff.exists())

# TI class from img_diff sign: <0 decrease(1), 0 stable(2), >0 increase(3)
def ti_class(d):
    c = np.full(d.shape, 2, np.int8); c[d > 0] = 3; c[d < 0] = 1; return c

# TS class from z-score at threshold thr
def ts_class(z, thr):
    c = np.full(z.shape, 2, np.int8); c[z > thr] = 3; c[z < -thr] = 1; return c

for thr in (0.0, 0.1, 0.25, 0.5, 0.75, 1.0, 1.5):
    agree = total = 0
    with rasterio.open(ts_z_path) as sz, rasterio.open(ti_diff) as st:
        for _, w in sz.block_windows(1):
            z = sz.read(1, window=w, masked=True)
            d = st.read(1, window=w, masked=True)
            valid = (~np.ma.getmaskarray(z)) & (~np.ma.getmaskarray(d))
            if not valid.any():
                continue
            zc = ts_class(np.ma.getdata(z), thr)[valid]
            dc = ti_class(np.ma.getdata(d))[valid]
            total += zc.size
            agree += int((zc == dc).sum())
    print(f"TS threshold +/-{thr}: agreement {agree/total:.1%}  (n={total:,})")


In [ ]:
# compared ts and ti
import numpy as np, rasterio
from pathlib import Path

ts_path = Path(r"./data/5_Typology/TimeSeries-1990-2020.tif")
ti_path = Path(r"./data/5_Typology/TimeInterval-1990-2020.tif")
print("TS exists:", ts_path.exists(), "| TI exists:", ti_path.exists())

# --- footprint comparison: how many valid pixels in each, and overlap ---
both = ts_only = ti_only = 0
agree = 0
with rasterio.open(ts_path) as a, rasterio.open(ti_path) as b:
    print("TS grid:", a.width, "x", a.height, "| TI grid:", b.width, "x", b.height)
    for _, w in a.block_windows(1):
        va_m = a.read(1, window=w, masked=True)
        vb_m = b.read(1, window=w, masked=True)
        va = ~np.ma.getmaskarray(va_m)      # TS valid
        vb = ~np.ma.getmaskarray(vb_m)      # TI valid
        both    += int((va & vb).sum())
        ts_only += int((va & ~vb).sum())
        ti_only += int((~va & vb).sum())
        # agreement only where BOTH valid
        v = va & vb
        if v.any():
            agree += int((np.ma.getdata(va_m)[v] == np.ma.getdata(vb_m)[v]).sum())

ts_total = both + ts_only
ti_total = both + ti_only
print(f"TS valid pixels : {ts_total:,}")
print(f"TI valid pixels : {ti_total:,}")
print(f"difference      : {ts_total - ti_total:,}")
print(f"both valid      : {both:,}")
print(f"TS-only (TI NoData here): {ts_only:,}")
print(f"TI-only (TS NoData here): {ti_only:,}")
if both:
    print(f"overall class agreement (on shared pixels): {agree/both:.1%}")

In [ ]:
#comparison tif to see the differnce between ts and ti 
import numpy as np, rasterio
from pathlib import Path

ts_path  = Path(r"./data/5_Typology/TimeSeries-1990-2020.tif")
ti_path  = Path(r"./data/5_Typology/TimeInterval-1990-2020.tif")
out_path = Path(r"./data/5_Typology/footprint_overlap.tif")

# Output codes:
#   0 = neither valid (background)   -> written as NoData
#   1 = both valid (overlap)
#   2 = TS only (TS has data, TI NoData)
#   3 = TI only (TI has data, TS NoData)

with rasterio.open(ts_path) as a, rasterio.open(ti_path) as b:
    assert (a.width, a.height) == (b.width, b.height), "grids differ!"
    profile = a.profile.copy()
    profile.update(count=1, dtype="uint8", nodata=0,
                   compress="deflate", tiled=True, blockxsize=512, blockysize=512)

    counts = {1: 0, 2: 0, 3: 0}
    with rasterio.open(out_path, "w", **profile) as dst:
        for _, w in a.block_windows(1):
            va = ~np.ma.getmaskarray(a.read(1, window=w, masked=True))   # TS valid
            vb = ~np.ma.getmaskarray(b.read(1, window=w, masked=True))   # TI valid
            out = np.zeros(va.shape, dtype=np.uint8)
            out[va & vb]   = 1   # both
            out[va & ~vb]  = 2   # TS only
            out[~va & vb]  = 3   # TI only
            dst.write(out, 1, window=w)
            counts[1] += int((out == 1).sum())
            counts[2] += int((out == 2).sum())
            counts[3] += int((out == 3).sum())

print(f"wrote: {out_path}")
print(f"  both valid (1): {counts[1]:,}")
print(f"  TS only   (2): {counts[2]:,}")
print(f"  TI only   (3): {counts[3]:,}")


In [ ]:
import numpy as np, rasterio, glob, random
from pathlib import Path
from rasterio.windows import Window

fp_path = Path(r"./data/5_Typology/footprint_overlap.tif")   # 3 = TI-only
year_glob = r"./data/3_MovingWindow/mw_area/*_area_1km.tif"
MIN_OBS = 10

year_paths = sorted(glob.glob(year_glob))
print(f"annual rasters found: {len(year_paths)}")
if not year_paths:
    raise FileNotFoundError("No annual rasters matched — check the mw_area path/pattern")

# --- 1) collect a sample of TI-only pixel locations (block by block, memory-safe) ---
ti_only = []
with rasterio.open(fp_path) as s:
    for _, w in s.block_windows(1):
        a = s.read(1, window=w)
        rr, cc = np.where(a == 3)
        for i in range(len(rr)):
            ti_only.append((w.row_off + int(rr[i]), w.col_off + int(cc[i])))
        if len(ti_only) >= 5000:
            break
print(f"TI-only pixels sampled: {len(ti_only)}")
picks = random.sample(ti_only, min(10, len(ti_only)))

# --- 2) read the 31-year series at each sampled pixel (1x1 windows = tiny) ---
valid_year_counts = []
srcs = [rasterio.open(p) for p in year_paths]
try:
    for (r, c) in picks:
        series = []
        for s in srcs:
            v = s.read(1, window=Window(c, r, 1, 1), masked=True)
            if np.ma.getmaskarray(v)[0, 0]:
                series.append(None)                         # NoData / missing
            else:
                series.append(round(float(np.ma.getdata(v)[0, 0]), 2))
        nvalid = sum(1 for x in series if x is not None)    # finite years (incl real 0s)
        valid_year_counts.append(nvalid)
        flag = "<10 -> TS drops" if nvalid < MIN_OBS else ">=10 -> TS should KEEP (!)"
        print(f"pixel ({r},{c}): {nvalid}/31 valid years  {flag}")
        print(f"   series: {series}")
finally:
    for s in srcs:
        s.close()

# --- 3) verdict ---
print("--- SUMMARY ---")
import statistics
print(f"sampled TI-only pixels: {len(valid_year_counts)}")
print(f"valid-year counts: min={min(valid_year_counts)}, "
      f"median={statistics.median(valid_year_counts)}, max={max(valid_year_counts)}")
under = sum(1 for n in valid_year_counts if n < MIN_OBS)
print(f"{under}/{len(valid_year_counts)} have < {MIN_OBS} valid years")
if under == len(valid_year_counts):
    print(">>> CONFIRMED: TI-only pixels fail the min_observations rule (hypothesis correct).")
elif under == 0:
    print(">>> HYPOTHESIS WRONG: these pixels have >=10 valid years, so min_observations is NOT why TS dropped them. Cause is something else.")
else:
    print(">>> MIXED: some fail min_observations, some don't -> more than one cause.")


In [ ]:
import rasterio, numpy as np
from rasterio.windows import Window
from pathlib import Path

r, c = 10258, 72796   # a confirmed TI-only pixel with full data

stages = [
    ("area z-score", r"./data/4_TS_and_TI/TS_zscore/area/area_MK_tau_z.tif"),
    ("edge z-score", r"./data/4_TS_and_TI/TS_zscore/edge/edge_MK_tau_z.tif"),
    ("pn z-score",   r"./data/4_TS_and_TI/TS_zscore/pn/pn_MK_tau_z.tif"),
    ("TS final typology", r"./data/5_Typology/TimeSeries-1990-2020.tif"),
]
for name, p in stages:
    p = Path(p)
    if not p.exists():
        print(f"{name:20}: FILE MISSING ({p})"); continue
    with rasterio.open(p) as s:
        v = s.read(1, window=Window(c, r, 1, 1), masked=True)
        if np.ma.getmaskarray(v)[0,0]:
            print(f"{name:20}: NoData  <-- drops out here")
        else:
            print(f"{name:20}: {float(np.ma.getdata(v)[0,0])}")

In [ ]:
import numpy as np, rasterio
from collections import Counter
from rasterio.windows import Window
from pathlib import Path

fp_path = Path(r"./data/5_Typology/footprint_overlap.tif")          # 3 = TI-only
ti_path = Path(r"./data/5_Typology/TimeInterval-1990-2020.tif")

ti_vals = Counter()
total = 0
with rasterio.open(fp_path) as fp, rasterio.open(ti_path) as ti:
    for _, w in fp.block_windows(1):
        f = fp.read(1, window=w)                 # footprint codes
        t = ti.read(1, window=w, masked=True)    # TI typology
        mask = (f == 3)                          # TI-only pixels
        if not mask.any():
            continue
        tv = np.ma.getdata(t)[mask]
        tm = np.ma.getmaskarray(t)[mask]
        for code in tv[~tm]:                      # TI classes present where TS is NoData
            ti_vals[int(code)] += 1
        total += int(mask.sum())

print(f"TI-only pixels: {total:,}")
print("TI typology values at TI-only locations (class: pixel count):")
for code, n in ti_vals.most_common():
    print(f"  class {code}: {n:,}  ({n/total:.1%})")


In [ ]:
import numpy as np, rasterio, glob
from collections import Counter
from rasterio.windows import Window
from pathlib import Path

fp = Path(r"./data/5_Typology/footprint_overlap.tif")
ti = Path(r"./data/5_Typology/TimeInterval-1990-2020.tif")

# find TI-only pixels that TI classifies as Dissection (8) or Shrinkage (3)
picks = []
with rasterio.open(fp) as f, rasterio.open(ti) as t:
    for _, w in f.block_windows(1):
        fa = f.read(1, window=w); ta = t.read(1, window=w, masked=True)
        m = (fa == 3) & (~np.ma.getmaskarray(ta)) & np.isin(np.ma.getdata(ta), [8, 3])
        rr, cc = np.where(m)
        for i in range(len(rr)):
            picks.append((w.row_off+int(rr[i]), w.col_off+int(cc[i]),
                          int(np.ma.getdata(ta)[rr[i], cc[i]])))
        if len(picks) >= 500: break

import random
for (r, c, ticlass) in random.sample(picks, min(6, len(picks))):
    # the three metric series
    for metric in ["area"]:   # start with area
        yps = sorted(glob.glob(f"./data/3_MovingWindow/mw_{metric}/*_{metric}_1km.tif"))
        series = []
        for p in yps:
            with rasterio.open(p) as s:
                v = s.read(1, window=Window(c, r, 1, 1), masked=True)
                series.append(None if np.ma.getmaskarray(v)[0,0] else round(float(np.ma.getdata(v)[0,0]),1))
    nz = sum(1 for x in series if x not in (None, 0))
    print(f"pixel ({r},{c}) TI={ticlass}: {series}")
    print(f"   nonzero years: {nz}, ends at: {series[-3:]}")


In [ ]:
import numpy as np, rasterio
from rasterio.windows import Window
from pathlib import Path

r, c = 10258, 72796   # confirmed TI-only pixel, valid z-scores, NoData final

stages = [
    ("area z-score", r"./data/4_TS_and_TI/TS_zscore/area/area_MK_tau_z.tif"),
    ("area remapped", r"./data/5_Typology/remap_timeseries/area_TS_rmp.tif"),
    ("edge remapped", r"./data/5_Typology/remap_timeseries/edge_TS_rmp.tif"),
    ("pn remapped",   r"./data/5_Typology/remap_timeseries/pn_TS_rmp.tif"),
    ("TS combined",   r"./data/5_Typology/combined_timeseries/combined.tif"),
]
for name, p in stages:
    p = Path(p)
    if not p.exists():
        print(f"{name:16}: FILE MISSING ({p})"); continue
    with rasterio.open(p) as s:
        v = s.read(1, window=Window(c, r, 1, 1), masked=True)
        print(f"{name:16}: {'NoData' if np.ma.getmaskarray(v)[0,0] else float(np.ma.getdata(v)[0,0])}")


In [ ]:
import numpy as np, rasterio, glob
from rasterio.windows import Window
from pathlib import Path
import random

fp = Path(r"./data/5_Typology/footprint_overlap.tif")             # 3 = TI-only
ti = Path(r"./data/5_Typology/TimeInterval-1990-2020.tif")

ti_labels = {1:"Attrition",2:"Aggregation",3:"Shrinkage",4:"Perforation",
             5:"Deformation",6:"Persistent",7:"Enlargement",8:"Dissection",
             9:"Frag per se",10:"Creation"}

# collect TI-only pixels with their TI class
picks = []
with rasterio.open(fp) as f, rasterio.open(ti) as t:
    for _, w in f.block_windows(1):
        fa = f.read(1, window=w); ta = t.read(1, window=w, masked=True)
        m = (fa == 3) & (~np.ma.getmaskarray(ta))
        rr, cc = np.where(m)
        for i in range(len(rr)):
            picks.append((w.row_off+int(rr[i]), w.col_off+int(cc[i]),
                          int(np.ma.getdata(ta)[rr[i],cc[i]])))
        if len(picks) >= 8000: break

sample = random.sample(picks, min(6, len(picks)))

metrics = ["area","edge","pn"]
series_files = {m: sorted(glob.glob(f"./data/3_MovingWindow/mw_{m}/*_{m}_1km.tif")) for m in metrics}

for (r, c, ticlass) in sample:
    print(f"=== pixel ({r},{c})  TI = {ticlass} ({ti_labels.get(ticlass,'?')}) | TS = NoData ===")
    for m in metrics:
        s = []
        for p in series_files[m]:
            with rasterio.open(p) as src:
                v = src.read(1, window=Window(c, r, 1, 1), masked=True)
                s.append(None if np.ma.getmaskarray(v)[0,0] else round(float(np.ma.getdata(v)[0,0]),1))
        print(f"  {m:5}: start={s[0]} end={s[-1]}  net={None if (s[0] is None or s[-1] is None) else round(s[-1]-s[0],1)}")
        print(f"         {s}")


In [ ]:
import numpy as np, rasterio, glob
from pathlib import Path

fp = Path(r"./data/5_Typology/footprint_overlap.tif")   # 1=both, 2=TS-only, 3=TI-only
metrics = ["area", "edge", "pn"]
year_files = {m: sorted(glob.glob(f"./data/3_MovingWindow/mw_{m}/*_{m}_1km.tif")) for m in metrics}
z_paths = {m: Path(f"./data/4_TS_and_TI/TS_zscore/{m}/{m}_MK_tau_z.tif") for m in metrics}
THR = 0.5
for m in metrics:
    print(f"{m}: {len(year_files[m])} yearly rasters | z exists: {z_paths[m].exists()}")

# accumulators per group (2=TS-only, 3=TI-only), per metric
def new_metric_acc():
    return {"n_series":0,"net_sum":0.0,"abs_net_sum":0.0,"nochange":0,"small_net":0,"start_sum":0.0,
            "n_z":0,"z_within":0,"z_pos":0,"z_neg":0,"z_sum":0.0,"z_sq":0.0,"z_nodata":0}
stats = {g: {"n":0, **{m:new_metric_acc() for m in metrics}} for g in (2,3)}

srcs = {m: [rasterio.open(p) for p in year_files[m]] for m in metrics}
zsrc = {m: rasterio.open(z_paths[m]) for m in metrics}
fpsrc = rasterio.open(fp)
try:
    for _, w in fpsrc.block_windows(1):
        fa = fpsrc.read(1, window=w)
        for g in (2,3):
            gmask = (fa == g)
            if not gmask.any():
                continue
            stats[g]["n"] += int(gmask.sum())
            for m in metrics:
                s = stats[g][m]
                # --- net change (endpoints) ---
                first = srcs[m][0].read(1, window=w, masked=True)
                last  = srcs[m][-1].read(1, window=w, masked=True)
                f0 = np.ma.getdata(first).astype(np.float64)
                f1 = np.ma.getdata(last).astype(np.float64)
                vok = gmask & (~np.ma.getmaskarray(first)) & (~np.ma.getmaskarray(last))
                if vok.any():
                    net = (f1 - f0)[vok]; start = f0[vok]
                    s["n_series"]   += net.size
                    s["net_sum"]    += float(net.sum())
                    s["abs_net_sum"]+= float(np.abs(net).sum())
                    s["start_sum"]  += float(start.sum())
                    s["nochange"]   += int((net == 0).sum())
                    with np.errstate(divide="ignore", invalid="ignore"):
                        frac = np.where(start != 0, np.abs(net)/start, np.inf)
                    s["small_net"]  += int((frac < 0.10).sum())
                # --- z-score distribution ---
                za = zsrc[m].read(1, window=w, masked=True)
                zm = np.ma.getmaskarray(za)[gmask]
                zv = np.ma.getdata(za)[gmask]
                s["z_nodata"] += int(zm.sum())
                zv = zv[~zm]
                if zv.size:
                    s["n_z"]     += zv.size
                    s["z_within"]+= int((np.abs(zv) < THR).sum())
                    s["z_pos"]   += int((zv >  THR).sum())
                    s["z_neg"]   += int((zv < -THR).sum())
                    s["z_sum"]   += float(zv.sum()); s["z_sq"] += float((zv*zv).sum())
finally:
    for m in metrics:
        for s_ in srcs[m]: s_.close()
        zsrc[m].close()
    fpsrc.close()

names = {2:"TS-only (TS classifies, TI NoData)", 3:"TI-only (TI classifies, TS NoData)"}
for g in (2,3):
    n = stats[g]["n"]
    print(f"===== {names[g]} =====")
    print(f"pixels: {n:,}  ({n*0.0009:,.0f} km2)")
    if n == 0: continue
    for m in metrics:
        s = stats[g][m]
        ns, nz = max(s["n_series"],1), max(s["n_z"],1)
        print(f"  [{m}] net change: mean={s['net_sum']/ns:+.2f} mean|net|={s['abs_net_sum']/ns:.2f} "
              f"nochange={s['nochange']/ns:.1%} |net|<10%start={s['small_net']/ns:.1%}")
        zmean = s["z_sum"]/nz; zstd = (s["z_sq"]/nz - zmean**2)**0.5
        print(f"       z-score   : mean={zmean:+.3f} std={zstd:.3f}  "
              f"|z|<{THR}(stable)={s['z_within']/nz:.1%} "
              f"z>+{THR}={s['z_pos']/nz:.1%} z<-{THR}={s['z_neg']/nz:.1%} "
              f"(z-NoData={s['z_nodata']:,})")


In [ ]:
import numpy as np, rasterio
from collections import Counter
from pathlib import Path

fp = Path(r"./data/5_Typology/footprint_overlap.tif")   # 3 = TI-only (TS NoData in final)
ts_combined = Path(r"./data/5_Typology/combined_timeseries/combined.tif")  # <-- adjust to your real TS combined path

# confirm grids line up first
with rasterio.open(fp) as a, rasterio.open(ts_combined) as b:
    print("fp grid:", a.width, a.height, "| combined grid:", b.width, b.height,
          "| aligned:", (a.width, a.height) == (b.width, b.height))

codes = Counter()
nodata = 0
with rasterio.open(fp) as f, rasterio.open(ts_combined) as c:
    for _, w in f.block_windows(1):
        fa = f.read(1, window=w)
        ca = c.read(1, window=w, masked=True)
        m = (fa == 3)
        if not m.any():
            continue
        cm = np.ma.getmaskarray(ca)[m]
        cv = np.ma.getdata(ca)[m]
        nodata += int(cm.sum())
        for code in cv[~cm]:
            codes[int(code)] += 1

print(f"TI-only pixels NoData in TS combined: {nodata:,}")
print("TS combined remap values at TI-only locations (value: count, km2):")
for code, n in codes.most_common():
    print(f"  {code}: {n:,}  ({n*0.0009:,.0f} km2)")


In [ ]:
#investigating ti more
import numpy as np, rasterio, glob, random
from rasterio.windows import Window
from pathlib import Path

fp = Path(r"./data/5_Typology/footprint_overlap.tif")
ti_diff = Path(r"./data/4_TS_and_TI/TI/img_diff/90-20_area.tif")
files = sorted(glob.glob("./data/3_MovingWindow/mw_area/*_area_1km.tif"))
y1990, y2020 = files[0], files[-1]

picks = []
with rasterio.open(fp) as f, rasterio.open(ti_diff) as d:
    for _, w in f.block_windows(1):
        fa = f.read(1, window=w); dv = d.read(1, window=w, masked=True)
        m = (fa == 2) & np.ma.getmaskarray(dv)   # TS-only AND img_diff NoData
        rr, cc = np.where(m)
        for i in range(len(rr)):
            picks.append((w.row_off+int(rr[i]), w.col_off+int(cc[i])))
        if len(picks) >= 3000: break

print(f"found {len(picks)} TS-only + img_diff-NoData pixels")
for (r, c) in random.sample(picks, min(10, len(picks))):
    with rasterio.open(y1990) as a, rasterio.open(y2020) as b:
        v90 = a.read(1, window=Window(c,r,1,1), masked=True)
        v20 = b.read(1, window=Window(c,r,1,1), masked=True)
    f90 = "NoData" if np.ma.getmaskarray(v90)[0,0] else float(np.ma.getdata(v90)[0,0])
    f20 = "NoData" if np.ma.getmaskarray(v20)[0,0] else float(np.ma.getdata(v20)[0,0])
    print(f"({r},{c}): 1990={f90}  2020={f20}")

